In [2]:
# Import libraries and set project paths

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'

print("Libraries and project paths loaded successfully.")

Libraries and project paths loaded successfully.


In [4]:
# Mount Google Drive

from google.colab import drive

drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [5]:
# Load the A-8 test channel

channel_name = "A-8"

data_A8 = np.load(test_path + "/" + channel_name + ".npy")

print("Channel:", channel_name)
print("Shape:", data_A8.shape)
print("Number of columns:", data_A8.shape[1])

print("\nFirst 5 rows:")
print(data_A8[:5])

Channel: A-8
Shape: (8375, 25)
Number of columns: 25

First 5 rows:
[[0.9392753 0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.       ]
 [0.9392753 0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.       ]
 [0.9392753 0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.       ]
 [0.9392753 0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0

In [6]:
# Check how much each column varies

for i in range(data_A8.shape[1]):
    column_std = np.std(data_A8[:, i])
    unique_values = len(np.unique(data_A8[:, i]))

    print(
        f"Column {i}: "
        f"std={column_std:.4f}, "
        f"unique values={unique_values}"
    )

Column 0: std=0.3766, unique values=34
Column 1: std=0.0607, unique values=2
Column 2: std=0.0450, unique values=2
Column 3: std=0.0523, unique values=2
Column 4: std=0.0450, unique values=2
Column 5: std=0.0763, unique values=2
Column 6: std=0.0698, unique values=2
Column 7: std=0.0000, unique values=1
Column 8: std=0.0000, unique values=1
Column 9: std=0.0000, unique values=1
Column 10: std=0.0000, unique values=1
Column 11: std=0.0000, unique values=1
Column 12: std=0.0000, unique values=1
Column 13: std=0.0000, unique values=1
Column 14: std=0.0000, unique values=1
Column 15: std=0.0000, unique values=1
Column 16: std=0.0000, unique values=1
Column 17: std=0.0000, unique values=1
Column 18: std=0.0000, unique values=1
Column 19: std=0.0362, unique values=2
Column 20: std=0.0000, unique values=1
Column 21: std=0.0626, unique values=2
Column 22: std=0.0476, unique values=2
Column 23: std=0.0000, unique values=1
Column 24: std=0.0000, unique values=1


In [7]:
# Inspect unique values in columns 1-24

for i in range(1, data_A8.shape[1]):
    unique_values = np.unique(data_A8[:, i])

    if len(unique_values) <= 5:
        print(f"Column {i}: {unique_values}")

Column 1: [0. 1.]
Column 2: [0. 1.]
Column 3: [0. 1.]
Column 4: [0. 1.]
Column 5: [0. 1.]
Column 6: [0. 1.]
Column 7: [0.]
Column 8: [0.]
Column 9: [0.]
Column 10: [0.]
Column 11: [0.]
Column 12: [0.]
Column 13: [0.]
Column 14: [0.]
Column 15: [0.]
Column 16: [0.]
Column 17: [0.]
Column 18: [0.]
Column 19: [0. 1.]
Column 20: [0.]
Column 21: [0. 1.]
Column 22: [0. 1.]
Column 23: [0.]
Column 24: [0.]


In [8]:
# List the files in the dataset archive

archive_path = project_path + '/data/raw/archive'

for root, dirs, files in os.walk(archive_path):
    for file in files:
        print(os.path.join(root, file))

/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/labeled_anomalies.csv
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/params.log
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/models/A-1.h5
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/models/C-1.h5
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/models/A-2.h5
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/models/A-7.h5
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/models/C-2.h5
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/models/A-9.h5
/content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/2018-05-19_15.00.10/models/A-8.h5
/content

In [9]:
# Check the number of columns across all training channels

feature_counts = {}

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        data = np.load(train_path + "/" + file)
        feature_counts[file.replace(".npy", "")] = data.shape[1]

from collections import Counter

column_counts = Counter(feature_counts.values())

print("Number of columns used by channels:")
for columns, count in sorted(column_counts.items()):
    print(f"{columns} columns -> {count} channels")

Number of columns used by channels:
25 columns -> 54 channels
55 columns -> 28 channels


In [10]:
# Check spacecraft information from labeled_anomalies.csv

labels_path = archive_path + "/labeled_anomalies.csv"

labels_df = pd.read_csv(labels_path)

print("Columns in labeled_anomalies.csv:")
print(labels_df.columns.tolist())

print("\nFirst 5 rows:")
print(labels_df.head())

Columns in labeled_anomalies.csv:
['chan_id', 'spacecraft', 'anomaly_sequences', 'class', 'num_values']

First 5 rows:
  chan_id spacecraft                           anomaly_sequences  \
0     P-1       SMAP  [[2149, 2349], [4536, 4844], [3539, 3779]]   
1     S-1       SMAP                              [[5300, 5747]]   
2     E-1       SMAP                [[5000, 5030], [5610, 6086]]   
3     E-2       SMAP                              [[5598, 6995]]   
4     E-3       SMAP                              [[5094, 8306]]   

                                  class  num_values  
0  [contextual, contextual, contextual]        8505  
1                               [point]        7331  
2              [contextual, contextual]        8516  
3                               [point]        8532  
4                               [point]        8307  


In [11]:
# Count channels by spacecraft

spacecraft_counts = labels_df["spacecraft"].value_counts()

print("Channels by spacecraft:")
print(spacecraft_counts)

Channels by spacecraft:
spacecraft
SMAP    55
MSL     27
Name: count, dtype: int64


In [12]:
# Check anomaly classes

print("Anomaly class examples:")
for i in range(5):
    print(labels_df.loc[i, "chan_id"], "->", labels_df.loc[i, "class"])

Anomaly class examples:
P-1 -> [contextual, contextual, contextual]
S-1 -> [point]
E-1 -> [contextual, contextual]
E-2 -> [point]
E-3 -> [point]


In [13]:
# Count anomaly types

all_classes = []

for classes in labels_df["class"]:
    all_classes.extend(
        classes.strip("[]").replace("'", "").split(", ")
    )

from collections import Counter

class_counts = Counter(all_classes)

print("Anomaly class counts:")
for anomaly_class, count in class_counts.items():
    print(f"{anomaly_class}: {count}")

Anomaly class counts:
contextual: 43
point: 62


In [14]:
# Count anomaly sequences by spacecraft

anomaly_sequence_counts = {}

for spacecraft in labels_df["spacecraft"].unique():
    spacecraft_data = labels_df[labels_df["spacecraft"] == spacecraft]

    total_sequences = 0

    for sequences in spacecraft_data["anomaly_sequences"]:
        total_sequences += len(eval(sequences))

    anomaly_sequence_counts[spacecraft] = total_sequences

print("Anomaly sequences by spacecraft:")

for spacecraft, count in anomaly_sequence_counts.items():
    print(f"{spacecraft}: {count}")

Anomaly sequences by spacecraft:
SMAP: 69
MSL: 36


In [15]:
# Check whether metadata lengths match the actual test files

mismatches = []

for _, row in labels_df.iterrows():
    channel = row["chan_id"]
    expected_length = int(row["num_values"])

    file_path = test_path + "/" + channel + ".npy"
    actual_length = np.load(file_path).shape[0]

    if expected_length != actual_length:
        mismatches.append(
            (channel, expected_length, actual_length)
        )

print("Number of mismatches:", len(mismatches))

if mismatches:
    print("\nMismatches:")
    for item in mismatches:
        print(item)
else:
    print("All channel lengths match the metadata.")

Number of mismatches: 0
All channel lengths match the metadata.


In [16]:
# Count channels with labeled anomalies

channels_with_anomalies = labels_df["chan_id"].nunique()

print("Channels with labeled anomalies:", channels_with_anomalies)
print("Total channels:", len(labels_df))

Channels with labeled anomalies: 81
Total channels: 82


#github push

In [17]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (8/8), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/06_load_dataset.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/07_understand_dataset.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
